# 3Blue1Brown — Deep Learning Series · Compiled Notes

*Complete lecture notes for the 4-part 3Blue1Brown "Neural Networks" series. Concise, early-graduate level. Each chapter below is the full content of its standalone notebook, compiled into one master file.*

**Running example throughout:** a multilayer perceptron (MLP) that recognizes handwritten MNIST digits — 784 inputs → 16 → 16 → 10 outputs, ~13,000 weights & biases.

## Contents
1. **Ch.1 — Structure.** What neurons, layers, weights, biases & the sigmoid are; the network as one function.
2. **Ch.2 — Gradient descent.** Learning = minimizing a cost function; the gradient as steepest descent.
3. **Ch.3 — Backprop intuition.** Distributing "blame" backward; nudges, fire-together-wire-together, SGD.
4. **Ch.4 — Backprop calculus.** The chain rule made formal; the derivatives that build $\nabla C$.

## The one-paragraph arc
A network is **one big function** (Ch.1) whose ~13,000 knobs we tune by **minimizing a cost** via gradient descent (Ch.2). Computing the needed gradient efficiently is **backpropagation** — intuitively, pushing desired "nudges" backward through the layers (Ch.3); formally, the **chain rule applied recursively** (Ch.4).

---
# Ch.1 — But what is a Neural Network?

<sub>*(compiled from `notes_ch1_network_structure.ipynb`)*</sub>

# Ch.1 — But what is a Neural Network?

*3Blue1Brown, Deep Learning series, Ch.1. Concise lecture notes. This chapter = **structure only**; learning is Ch.2.*

**Running task:** recognize handwritten digits from a 28×28 grayscale image. Trivial for a brain, hard to hand-code → motivates neural nets.

## Neurons

- A **neuron** = a thing holding a number between **0 and 1**, called its **activation** (lit up when high).
- We use the *plain vanilla* multilayer perceptron — prerequisite for every fancier modern variant.

## Layers

| Layer | # neurons | meaning |
|---|---|---|
| Input | **784** = 28×28 | grayscale of each pixel (0 black → 1 white) |
| Hidden 1 | 16 | (arbitrary choice) |
| Hidden 2 | 16 | (arbitrary choice) |
| Output | **10** | one per digit; activation = how much net "thinks" it's that digit |

- **Brightest output neuron = the network's guess.**
- Core idea: **activations in one layer determine activations in the next.** Loosely analogous to neurons firing → causing others to fire.

## Why a layered structure? (the *hope*)

Recognition decomposes into **layers of abstraction**:

> pixels → **edges** → **patterns/subcomponents** (loops, lines) → **digits**

- e.g. a 9 = loop on top + vertical line. Hope: a hidden neuron fires for any "loop up top."
- Same abstraction idea generalizes (e.g. speech: audio → sounds → syllables → words).
- ⚠️ Whether the trained net *actually* does this is an open question (revisited in Ch.2).

## How one layer drives the next — weights & biases

For a single neuron in the next layer:

1. **Weight** $w_i$ on each connection from the previous layer (just numbers; green = +, red = −).
2. Compute the **weighted sum** of incoming activations $a_i$.
   - Tuning weights = choosing *what pixel pattern* the neuron looks for (e.g. + weights on a region, − weights around it → fires for an **edge**).
3. Add a **bias** $b$ — shifts *how high* the sum must be before the neuron meaningfully activates.
4. **Squish** into (0,1) with the **sigmoid** (logistic) so output is a valid activation.

$$a^{(1)}_j = \sigma\!\left(\sum_{i} w_{ji}\, a^{(0)}_i + b_j\right), \qquad \sigma(x)=\frac{1}{1+e^{-x}}$$

- **Weights → what pattern.** **Bias → activation threshold.**

## Compact matrix form

Stack activations into a vector, weights into a matrix (one **row per next-layer neuron**), biases into a vector:

$$\mathbf{a}^{(1)} = \sigma\!\left(\mathbf{W}\,\mathbf{a}^{(0)} + \mathbf{b}\right)$$

- $\sigma$ is applied **componentwise**.
- Why bother: tight notation **+** fast code (libraries heavily optimize matrix–vector products). *(See 3B1B Linear Algebra series, Ch.3, for the geometry.)*

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# one layer transition: 784 inputs -> 16 neurons
a0 = np.random.rand(784)        # input activations in [0,1]
W  = np.random.randn(16, 784)   # one row per next-layer neuron
b  = np.random.randn(16)

a1 = sigmoid(W @ a0 + b)        # -> 16 activations in (0,1)
a1.shape

## The big picture

- A neuron is really a **function** of the previous layer's outputs; the **whole network is one function**: 784 numbers in → 10 numbers out.
- This net has **~13,000 weights + biases** = the "knobs and dials."
- **Learning** (Ch.2) = an algorithm finding a setting of those ~13,000 parameters that solves the task.

## Footnote: sigmoid is old-school (interview w/ Lisha Li)

- Sigmoid came from the biological "inactive vs active" analogy, but **modern deep nets rarely use it**.
- **ReLU** = $\max(0, a)$ is much easier to train on very deep networks → became the default.

$$\text{ReLU}(a) = \max(0, a)$$

---
# Ch.2 — Gradient Descent: how networks learn

<sub>*(compiled from `notes_ch2_gradient_descent.ipynb`)*</sub>

# Ch.2 — Gradient Descent: how neural networks learn

*3Blue1Brown, Deep Learning series, Ch.2. Concise lecture notes. Ch.1 built the **structure**; this chapter is **how the ~13,000 weights & biases get set**.*

**Setup carried over:** MLP for MNIST digits — 784 inputs → 16 → 16 → 10 outputs. "Learning" = a calculus problem: **find the minimum of a cost function**.

## Learning = optimization

- The network starts with **random** weights & biases → output is garbage.
- We need a way to (a) **score** how bad it is, then (b) **adjust** the knobs to make it less bad.
- Framing: this is just **finding the minimum of a function** by calculus.

## The cost function (a.k.a. loss)

For **one** training example: compare the net's 10 output activations to the desired "one-hot" answer (e.g. the digit 3 → output 3 should be 1.0, all others 0.0).

$$C_{\text{example}} = \sum_{j=0}^{9}\left(a^{(L)}_j - y_j\right)^2$$

- **Small** sum = net is confident & correct. **Large** sum = net is confused.
- This is the **"lousiness"** measure of the network on that one example.
- The **total cost** = the **average** of $C_{\text{example}}$ over **all** training examples.

$$C(\mathbf{w}, \mathbf{b}) = \frac{1}{N}\sum_{k=1}^{N} C_{\text{example}_k}$$

> Input to the cost function = **all ~13,000 weights & biases.** Output = **one number** (badness). Its "parameters" = the entire training set.

## Minimizing a function — the calculus intuition

- A simple function of one variable: find the minimum by following the **slope** downhill — step **left** where slope > 0, **right** where slope < 0, step size ∝ |slope| so you **slow near a flat valley**.
- This avoids ever solving for the minimum explicitly (impossible for a 13,000-D function).
- ⚠️ **You land in a *local* minimum, not necessarily the global one** — which valley depends on where you start. There is no easy guarantee of finding the deepest valley.

## Gradient descent (the general case)

Generalize from 1 variable to **all ~13,000** at once. From multivariable calculus:

- The **gradient** $\nabla C$ is the vector of all partial derivatives — it points in the direction of **steepest ascent**.
- So $-\nabla C$ points **downhill**, the direction that **decreases cost fastest**.

**Algorithm:**
1. Compute $\nabla C$.
2. Take a small step: $\mathbf{w} \leftarrow \mathbf{w} - \eta\,\nabla C$  (learning rate $\eta$).
3. Repeat.

This nudges the weights & biases repeatedly toward a (local) minimum. **Computing $-\nabla C$ efficiently = backpropagation (Ch.3).**

## Interpreting the gradient vector

$\nabla C$ is a list of ~13,000 numbers — two things to read off each component:

- **Sign** → whether to nudge that weight/bias **up or down**.
- **Magnitude** → **how much it matters** — which adjustments give the most **"bang for your buck."**

> The gradient encodes the *relative importance* of every knob: a large component means that weight has a big effect on cost, so changing it is "worth more" than nudging a low-magnitude one.

In [ ]:
import numpy as np

# Toy gradient descent on a 1-D cost surface to build intuition.
def cost(x):       return (x - 3)**2 + 2          # min at x = 3
def grad(x):       return 2 * (x - 3)             # dC/dx

x   = 8.0          # random start
eta = 0.1          # learning rate (step size factor)
for step in range(25):
    x -= eta * grad(x)                            # w <- w - eta * grad
print(f"converged x = {x:.4f}  (true min = 3)")
print(f"cost = {cost(x):.4f}  (true min = 2)")

## What the network actually learns (generalization vs memorization)

- A trained net can hit **high accuracy** on test digits — but does it learn **edges → loops → digits** as Ch.1 *hoped*?
- Inspecting the weights of the second layer: they look **mostly like random noise**, not crisp edge detectors. The net found *a* minimum, but **not the "intelligent" structure** we imagined.
- Stress test: a net trained on **randomly-labeled** data can still drive cost to zero — i.e. it can **memorize** the dataset rather than learn a pattern. (Zhang et al., "Understanding deep learning requires rethinking generalization.")
- **Lesson:** low training cost ≠ understanding. Generalization is a separate, harder property.

## Modern context & references

- This is the **plain** gradient-descent picture. More modern/complex networks find **better, more efficient minima** — but these basics are the foundation for every modern architecture.
- A practical subtlety (foreshadowing Ch.3): real training uses **mini-batches / stochastic gradient descent** — estimate the gradient from a small random subset each step for speed.

**References**
- **MNIST** database — handwritten-digit benchmark.
- **Michael Nielsen**, *Neural Networks and Deep Learning* (free online book + code).
- **Chris Olah** blog; **Distill.pub** — visual deep-learning explainers.
- Research on **memorization / optimization landscapes** (Zhang et al. 2017; landscape-of-minima papers).

---
# Ch.3 — Backpropagation, intuitively

<sub>*(compiled from `notes_ch3_backpropagation.ipynb`)*</sub>

# Ch.3 — Backpropagation, intuitively

*3Blue1Brown, Deep Learning series, Ch.3. Concise lecture notes. Ch.2 said gradient descent needs $-\nabla C$; this chapter is **what that gradient means** and **how backprop computes it** — intuition, no calculus yet (that's Ch.4).*

**Carried over:** MLP for MNIST, ~13,000 weights & biases. Learning = nudge them to minimize cost.

## Recap: what the gradient means

- Cost $C$ takes all ~13,000 weights & biases → one "badness" number, averaged over training data.
- $-\nabla C$ tells us how to nudge **every** weight & bias to reduce cost fastest.
- Each component's **magnitude = sensitivity**: how much cost changes per unit change in that weight/bias → its **relative importance**.

> Backprop = the algorithm that **computes this gradient**. Goal here: get intuition for *why* it has the form it does.

## Step 1 — Focus on a single training example

Take one image (say a **'2'**). Current output (10 activations) is junk, e.g. `[0.5, 0.8, 0.2, 1.0, ...]`.

**Desired:** push output neuron **'2' → 1.0**, all other outputs **→ 0.0**.

The *sizes* of the desired nudges are **proportional to how far off** each output currently is — the more wrong a neuron, the harder we want to push it.

## Step 2 — Three ways to nudge an output neuron

To raise one output activation $a^{(L)}_j = \sigma(\dots)$, you can change three things feeding it:

1. **Bias** $b$ — shift it directly.
2. **Weights** $w$ — increase weights on connections from the **most active** previous neurons.
   - *Biggest bang for the buck*: changes to weights from high-activation neurons matter most (weight × activation). → **"neurons that fire together, wire together"** (loose Hebbian analogy).
3. **Previous activations** $a^{(L-1)}$ — you can't set these directly, but you can *want* them changed: neurons with **positive** weights should be **more** active, negative weights **less** active.

Point 3 is the key: it expresses a **desire for changes one layer back.**

## Step 3 — Add up the demands, then propagate backward

- Every one of the 10 output neurons has its **own** wishlist for how each previous-layer activation should change.
- **Sum these wishes** (weighted by how much each output cares) → a single list of desired changes for layer $L-1$.
- That list is exactly the same kind of "desired output" we started with — so **recurse**: apply the same logic to get desired changes for layer $L-2$, and so on **back to the input**.

This backward flow of "desired nudges" through every layer = **backpropagation**. It assigns **blame**: each weight & bias is told how to change.

## Step 4 — One example isn't enough: average

- The nudges above came from **one** training image — they'd make *that* digit better but ignore all others.
- The true desired nudge for each weight/bias = **average** of the nudges demanded by **every** training example.
- That averaged set of nudges **is** (proportional to) the negative gradient $-\nabla C$.

$$-\nabla C \;\propto\; \frac{1}{N}\sum_{k=1}^{N}(\text{nudges from example }k)$$

## Step 5 — Stochastic Gradient Descent (SGD)

Averaging over **all** N examples for **every** step is far too slow.

**Fix:** shuffle the data, split into **mini-batches** (e.g. 100 examples). Each step:
- compute the gradient on **one mini-batch** → an **approximation** of the true gradient,
- take a step, move to the next batch.

- Each step is **cheap & fast**; the path **zig-zags** downhill instead of taking the exact straightest route.
- Analogy: a **drunk man stumbling quickly downhill** beats a careful man calculating each exact step.
- This is **SGD** — the standard way real networks are trained.

In [ ]:
import numpy as np

# Illustrate the backprop intuition on a single neuron + the SGD averaging idea.
# Desired-nudge sizes are proportional to how wrong each output is.
output  = np.array([0.5, 0.8, 0.2, 1.0, 0.3, 0.6, 0.1, 0.0, 0.4, 0.7])
target  = np.zeros(10); target[2] = 1.0          # this image is a '2'

error          = output - target                 # how wrong each output neuron is
desired_nudge  = -error                           # push opposite the error
print("nudge on neuron '2':", round(desired_nudge[2], 2), "(big positive -> raise it)")
print("nudge on neuron '3':", round(desired_nudge[3], 2), "(negative -> suppress it)")

# Weight update favors connections from the most active previous neurons (w x a):
prev_act       = np.array([0.9, 0.1, 0.7])        # previous-layer activations
weight_nudge   = desired_nudge[2] * prev_act      # 'fire together, wire together'
print("weight nudges into '2':", np.round(weight_nudge, 2), "-> active neuron #0 gets biggest")

# SGD: gradient of a mini-batch approximates the full-data gradient.
rng        = np.random.default_rng(0)
full_grad  = rng.normal(size=1000)                # pretend: true gradient over all data
batch_grad = full_grad[:100]                      # one mini-batch estimate
print("full-data mean:", round(full_grad.mean(), 3),
      "| mini-batch mean:", round(batch_grad.mean(), 3), "(noisy but cheap)")

## Why this matters

- Backprop **distributes blame**: it efficiently turns "the output was wrong" into a precise "**this** weight should go up, **that** bias down" for all ~13,000 parameters.
- It's just the **chain rule applied recursively**, organized so each layer reuses the next layer's result — the explicit calculus is **Ch.4**.
- Combined with SGD, it's what makes training large networks **computationally feasible**.

**References**
- Michael Nielsen, *Neural Networks and Deep Learning* — Ch. on backprop.
- 3B1B **Ch.4** — backpropagation calculus (the chain-rule details).

---
# Ch.4 — Backpropagation calculus

<sub>*(compiled from `notes_ch4_backprop_calculus.ipynb`)*</sub>

# Ch.4 — Backpropagation calculus

*3Blue1Brown, Deep Learning series, Ch.4. Concise lecture notes. Ch.3 gave the **intuition**; this chapter makes it **formal** — the chain rule, the way ML people actually write it.*

**Goal:** for each weight & bias, find how **sensitive** the cost is to it — i.e. the partial derivative — since those partials *are* the gradient $\nabla C$. Expect some confusion; pause & ponder.

## The toy network: one neuron per layer

Strip it to the bone: every layer has a **single** neuron → 3 weights, 3 biases. Focus on the **last connection**.

**Notation** (superscripts $^{(L)}$ are layer *indices*, not exponents):

| Symbol | Meaning |
|---|---|
| $a^{(L)}$ | activation of the last neuron |
| $a^{(L-1)}$ | activation of the previous neuron |
| $w^{(L)},\ b^{(L)}$ | weight & bias on the last connection |
| $y$ | desired output for this training example (e.g. 0 or 1) |
| $z^{(L)}$ | the **weighted sum** (pre-activation) |

**Forward chain for one example:**

$$z^{(L)} = w^{(L)} a^{(L-1)} + b^{(L)}, \qquad a^{(L)} = \sigma\!\left(z^{(L)}\right), \qquad C_0 = \left(a^{(L)} - y\right)^2$$

Conceptually: $w, a^{(L-1)}, b \rightarrow z \rightarrow a \rightarrow C_0$. Each is just a number on its own little number line.

## The chain rule for $\partial C_0/\partial w^{(L)}$

A tiny nudge $\partial w^{(L)}$ nudges $z^{(L)}$, which nudges $a^{(L)}$, which nudges $C_0$. Multiply the three ratios:

$$\frac{\partial C_0}{\partial w^{(L)}} = \frac{\partial z^{(L)}}{\partial w^{(L)}}\,\frac{\partial a^{(L)}}{\partial z^{(L)}}\,\frac{\partial C_0}{\partial a^{(L)}}$$

**The three pieces:**

$$\frac{\partial C_0}{\partial a^{(L)}} = 2\left(a^{(L)} - y\right) \qquad \frac{\partial a^{(L)}}{\partial z^{(L)}} = \sigma'\!\left(z^{(L)}\right) \qquad \frac{\partial z^{(L)}}{\partial w^{(L)}} = a^{(L-1)}$$

**Read the meanings:**
- $2(a^{(L)}-y)$ → proportional to **how wrong** the output is; big error ⇒ big impact.
- $\sigma'(z^{(L)})$ → slope of the nonlinearity (sigmoid/ReLU).
- $a^{(L-1)}$ → a weight's influence scales with **how active the previous neuron is** → *"neurons that fire together, wire together."*

## From one example to the gradient

- The above is for **one** training example. Full cost = **average** over all examples, so:

$$\frac{\partial C}{\partial w^{(L)}} = \frac{1}{N}\sum_{k=1}^{N}\frac{\partial C_k}{\partial w^{(L)}}$$

- This is just **one component** of $\nabla C$ — but computing it is **>50% of the work**; the rest reuse the same pieces.

## Bias and the backward step

**Bias:** identical chain, swap $\dfrac{\partial z}{\partial w}$ for $\dfrac{\partial z}{\partial b} = 1$:

$$\frac{\partial C_0}{\partial b^{(L)}} = \frac{\partial a^{(L)}}{\partial z^{(L)}}\,\frac{\partial C_0}{\partial a^{(L)}} = \sigma'\!\left(z^{(L)}\right)\cdot 2\left(a^{(L)}-y\right)$$

**Propagating backward — the key piece:** sensitivity of $z^{(L)}$ to the *previous activation* is the weight itself:

$$\frac{\partial z^{(L)}}{\partial a^{(L-1)}} = w^{(L)}$$

We can't set $a^{(L-1)}$ directly, but this lets us **iterate the same chain rule one layer back** → and back, and back. That recursion *is* backpropagation.

## Multiple neurons per layer (the real case)

Surprisingly little changes — just **more indices**. Index layer $L-1$ with $k$, layer $L$ with $j$.

- **Cost** sums over output neurons: $\displaystyle C_0 = \sum_{j}\left(a^{(L)}_j - y_j\right)^2$
- **Weight** $w^{(L)}_{jk}$ connects neuron $k$ (layer $L-1$) → neuron $j$ (layer $L$). *(Order $jk$ matches the weight-matrix convention from Ch.1.)*
- $z^{(L)}_j = \sum_k w^{(L)}_{jk}\,a^{(L-1)}_k + b^{(L)}_j$, and $a^{(L)}_j = \sigma(z^{(L)}_j)$.

The chain-rule expression for $\partial C_0/\partial w^{(L)}_{jk}$ looks **essentially the same** as the one-neuron case.

**What's genuinely different:** a neuron $a^{(L-1)}_k$ now feeds **every** neuron $j$ in the next layer, so it affects the cost through **multiple paths** — you must **sum over $j$**:

$$\frac{\partial C_0}{\partial a^{(L-1)}_k} = \sum_{j} \frac{\partial z^{(L)}_j}{\partial a^{(L-1)}_k}\,\frac{\partial a^{(L)}_j}{\partial z^{(L)}_j}\,\frac{\partial C_0}{\partial a^{(L)}_j}$$

Once you have this for layer $L-1$, **repeat the whole process** for the weights & biases feeding into it.

In [ ]:
import numpy as np

# Backprop on the toy 1-neuron-per-layer network: verify the chain rule numerically.
def sigmoid(z):  return 1/(1+np.exp(-z))
def dsigmoid(z): return sigmoid(z)*(1-sigmoid(z))

w, b, a_prev, y = 0.8, -0.5, 0.6, 1.0          # last-connection params + previous activation + target

z = w*a_prev + b                                # pre-activation
a = sigmoid(z)
C = (a - y)**2

# Analytic chain rule:  dC/dw = dz/dw * da/dz * dC/da
dC_da = 2*(a - y)
da_dz = dsigmoid(z)
dz_dw = a_prev
dC_dw = dz_dw * da_dz * dC_da
dC_db = 1     * da_dz * dC_da                    # dz/db = 1
dC_dap = w    * da_dz * dC_da                    # dz/da_prev = w  (the backward step)

# Numeric check via finite difference:
eps = 1e-6
num_dC_dw = (( (sigmoid((w+eps)*a_prev+b)-y)**2 ) - C) / eps
print(f"dC/dw analytic = {dC_dw:.6f}   numeric = {num_dC_dw:.6f}")
print(f"dC/db = {dC_db:.6f}   dC/da_prev (backprop signal) = {dC_dap:.6f}")

## The big picture

- These chain-rule expressions give **every component of $\nabla C$** — the derivatives gradient descent uses to step downhill.
- The whole algorithm = **chain rule applied recursively, layer by layer, backward**, reusing each layer's result for the one before it.
- That's backpropagation — the workhorse behind how neural networks learn. *Don't worry if it takes time to digest.*

**Series complete:** Ch.1 structure → Ch.2 gradient descent → Ch.3 backprop intuition → **Ch.4 backprop calculus.**

**References**
- 3B1B **Essence of Calculus** series — for the chain rule itself.
- Michael Nielsen, *Neural Networks and Deep Learning* — backprop chapter & equations.